In [1]:
!pip install pyspark -q


In [2]:
from google.colab import files
uploaded = files.upload()

Saving courses (1).csv to courses (1).csv
Saving progress_clean.csv to progress_clean.csv
Saving enrollments_clean.csv to enrollments_clean.csv


In [4]:
from pyspark.sql import SparkSession, functions as F

spark = (SparkSession.builder
         .appName("CourseAnalysis")
         .master("local[*]")
         .getOrCreate())


In [9]:
import os
os.rename("courses (1).csv", "courses.csv")
print(os.listdir("."))

['.config', 'enrollments_clean.csv', 'progress_clean.csv', 'courses.csv', 'sample_data']


In [11]:
enrollments = spark.read.csv("enrollments_clean.csv", header=True, inferSchema=True)
progress    = spark.read.csv("progress_clean.csv",    header=True, inferSchema=True)
courses     = spark.read.csv("courses.csv",           header=True, inferSchema=True)

print("enrollments:", enrollments.count())
print("progress   :", progress.count())
print("courses    :", courses.count())
enrollments.show(5)

enrollments: 624
progress   : 620
courses    : 12
+-------------+----------+---------+---------------+---------+
|enrollment_id|student_id|course_id|enrollment_date|   status|
+-------------+----------+---------+---------------+---------+
|            1|         2|        2|     2025-09-19|completed|
|            2|         2|        3|     2025-05-13|   active|
|            3|         2|        6|     2025-04-05|   active|
|            4|         3|       10|     2025-03-02|   active|
|            5|         4|        5|     2025-09-30|completed|
+-------------+----------+---------+---------------+---------+
only showing top 5 rows


In [12]:
enrollments.printSchema()
progress.show(5)
courses.show()
enrollments.groupBy("status").count().show()

root
 |-- enrollment_id: integer (nullable = true)
 |-- student_id: integer (nullable = true)
 |-- course_id: integer (nullable = true)
 |-- enrollment_date: date (nullable = true)
 |-- status: string (nullable = true)

+-------------+----------+---------+----------+------------+
|enrollment_id|student_id|course_id|completion|last_updated|
+-------------+----------+---------+----------+------------+
|            1|         2|        2|     100.0|  2025-09-22|
|            2|         2|        3|      47.5|  2025-07-06|
|            3|         2|        6|      47.2|  2025-05-15|
|            4|         3|       10|       6.7|  2025-03-16|
|            5|         4|        5|     100.0|  2025-11-16|
+-------------+----------+---------+----------+------------+
only showing top 5 rows
+---------+--------------------+------------+--------------+
|course_id|         course_name|    category|duration_hours|
+---------+--------------------+------------+--------------+
|        1|Python for Be

In [14]:
joined = (enrollments
          .join(progress.select("enrollment_id", "completion"), on="enrollment_id", how="inner")
          .join(courses.select("course_id", "course_name"),     on="course_id",     how="inner"))

print("Joined rows:", joined.count())
joined.select("enrollment_id", "student_id", "course_name", "status", "completion").show()

Joined rows: 620
+-------------+----------+--------------------+---------+----------+
|enrollment_id|student_id|         course_name|   status|completion|
+-------------+----------+--------------------+---------+----------+
|            1|         2|    SQL Fundamentals|completed|     100.0|
|            2|         2|  MongoDB Essentials|   active|      47.5|
|            3|         2|Azure Databricks ...|   active|      47.2|
|            4|         3|        Git & GitHub|   active|       6.7|
|            5|         4|      PySpark Basics|completed|     100.0|
|            6|         4|Azure Databricks ...|  dropped|       6.1|
|            7|         4|Azure DevOps Pipe...|completed|     100.0|
|            8|         5|    SQL Fundamentals|  dropped|      43.7|
|            9|         5|Azure Databricks ...|completed|     100.0|
|           10|         5|Machine Learning 101|  dropped|       0.9|
|           11|         6|Python for Beginners|completed|     100.0|
|           12|  

In [16]:
summary = (joined.groupBy("course_id", "course_name")
           .agg(F.count("*").alias("total_enrolled"),
                F.sum(F.when(F.col("status") == "completed", 1).otherwise(0)).alias("completed"),
                F.sum(F.when(F.col("status") == "dropped",   1).otherwise(0)).alias("dropped"),
                F.round(F.avg("completion"), 2).alias("avg_progress"))
           .withColumn("completion_rate_pct", F.round(100 * F.col("completed") / F.col("total_enrolled"), 1))
           .withColumn("dropout_rate_pct",    F.round(100 * F.col("dropped")   / F.col("total_enrolled"), 1)))

summary.orderBy(F.desc("total_enrolled")).show()

+---------+--------------------+--------------+---------+-------+------------+-------------------+----------------+
|course_id|         course_name|total_enrolled|completed|dropped|avg_progress|completion_rate_pct|dropout_rate_pct|
+---------+--------------------+--------------+---------+-------+------------+-------------------+----------------+
|        8|Machine Learning 101|            62|       25|     11|       63.74|               40.3|            17.7|
|        6|Azure Databricks ...|            61|       29|     11|       68.32|               47.5|            18.0|
|        7|Azure DevOps Pipe...|            60|       25|     15|       62.93|               41.7|            25.0|
|       11|Statistics for Da...|            57|       28|      9|       69.94|               49.1|            15.8|
|       10|        Git & GitHub|            54|       22|     17|       61.31|               40.7|            31.5|
|        3|  MongoDB Essentials|            54|       13|     16|       

In [18]:
top_completed = summary.orderBy(F.desc("completed")).limit(5)
top_completed.select("course_name", "completed", "completion_rate_pct").show()

+--------------------+---------+-------------------+
|         course_name|completed|completion_rate_pct|
+--------------------+---------+-------------------+
|Azure Databricks ...|       29|               47.5|
|Statistics for Da...|       28|               49.1|
| Power BI Dashboards|       28|               57.1|
|Machine Learning 101|       25|               40.3|
|Azure DevOps Pipe...|       25|               41.7|
+--------------------+---------+-------------------+



In [20]:
top_dropped = summary.orderBy(F.desc("dropped")).limit(5)
top_dropped.select("course_name", "dropped", "dropout_rate_pct").show()

+--------------------+-------+----------------+
|         course_name|dropped|dropout_rate_pct|
+--------------------+-------+----------------+
|        Git & GitHub|     17|            31.5|
|  MongoDB Essentials|     16|            29.6|
|Azure DevOps Pipe...|     15|            25.0|
|      PySpark Basics|     12|            22.2|
|Azure Databricks ...|     11|            18.0|
+--------------------+-------+----------------+



In [22]:
top_completed.toPandas().to_csv("top_completed.csv", index=False)
top_dropped.toPandas().to_csv("top_dropped.csv", index=False)

In [23]:
from google.colab import files

files.download("top_completed.csv")
files.download("top_dropped.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>